In [0]:
%sql

SHOW CATALOGS;

In [0]:
%sql
USE CATALOG mes;
USE SCHEMA bronze;

In [0]:
%sql
SELECT * 
FROM mes.bronze.fms_fab
LIMIT 20;

In [0]:
%sql
SELECT * 
FROM mes.bronze.fms_fac
LIMIT 20;

In [0]:
spark.sql("SHOW SCHEMAS IN mes").show()

In [0]:
display(spark.sql("SHOW SCHEMAS IN mes"))

In [0]:
# Lista desplegable
dbutils.widgets.dropdown(
    "load_type",
    "FULL",
    ["FULL", "INCREMENTAL"],
    "Load type"
)

# Varias opciones seleccionables
dbutils.widgets.multiselect(
    "layers",
    "bronze",
    ["raw", "bronze", "silver", "gold"],
    "Layers"
)

# Checkbox
dbutils.widgets.combobox(
    "reset_environment",
    "NO",
    ["YES", "NO"],
    "Reset environment"
)

In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
SELECT * FROM mes.bronze.ingestion_watermark

In [0]:
# Read MDT Logs Bronze Delta table
df_bronze = spark.table("mes.bronze.mdt_logs")

display(df_bronze)

In [0]:
# ============================================================
# SUPABASE REST API -> DATABRICKS SPARK DATAFRAME
# Example: FMS_FAB
# ============================================================

import requests


# ============================================================
# 1. CONFIGURATION
# ============================================================

SUPABASE_URL = "https://hamiefeokmgyuscjutmt.supabase.co"

SUPABASE_KEY = "sb_publishable_sjvKR1iD3HcU6ClAtEHw4w_OVJG0643"

TABLE_NAME = "fms_fab"


# ============================================================
# 2. BUILD REST API URL
# ============================================================

api_url = (
    f"{SUPABASE_URL}/rest/v1/{TABLE_NAME}"
)

print(f"API URL: {api_url}")


# ============================================================
# 3. HTTP HEADERS
# ============================================================

headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Accept": "application/json"
}


# ============================================================
# 4. QUERY PARAMETERS
# ============================================================

# Select all columns from fms_fab

params = {
    "select": "*"
}


# ============================================================
# 5. CALL SUPABASE REST API
# ============================================================

try:

    print(
        f"Calling Supabase REST API: {TABLE_NAME}"
    )

    response = requests.get(
        api_url,
        headers=headers,
        params=params,
        timeout=30
    )


    # --------------------------------------------------------
    # Show HTTP status
    # --------------------------------------------------------

    print(
        f"HTTP Status Code: {response.status_code}"
    )


    # --------------------------------------------------------
    # Raise exception for HTTP errors
    # 400, 401, 403, 404, 500, etc.
    # --------------------------------------------------------

    response.raise_for_status()


    # --------------------------------------------------------
    # Convert JSON response to Python object
    # --------------------------------------------------------

    data = response.json()


    print(
        f"Records received: {len(data)}"
    )


except requests.exceptions.RequestException as error:

    print(
        f"❌ REST API request failed: {error}"
    )

    raise


# ============================================================
# 6. VALIDATE DATA
# ============================================================

if not data:

    raise ValueError(
        f"No data returned from Supabase table: {TABLE_NAME}"
    )


print(
    "✅ Data successfully received from Supabase REST API"
)


# ============================================================
# 7. CREATE SPARK DATAFRAME
# ============================================================

df_fms_fab = spark.createDataFrame(data)


# ============================================================
# 8. DISPLAY DATAFRAME
# ============================================================

display(df_fms_fab)


# ============================================================
# 9. PRINT SPARK SCHEMA
# ============================================================

df_fms_fab.printSchema()


# ============================================================
# 10. RECORD COUNT
# ============================================================

record_count = df_fms_fab.count()

print(
    f"Spark DataFrame record count: {record_count}"
)


# ============================================================
# 11. WRITE TO UNITY CATALOG BRONZE TABLE
# ============================================================

TARGET_TABLE = "mes.bronze.fms_fab"


(
    df_fms_fab
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)


print(
    f"✅ Data successfully written to: {TARGET_TABLE}"
)

API URL: https://hamiefeokmgyuscjutmt.supabase.co/rest/v1/fms_fab
Calling Supabase REST API: fms_fab
HTTP Status Code: 200
Records received: 4
✅ Data successfully received from Supabase REST API


COUNTRY,CREATED_DATE,FAB_DESC,FAB_ID,FAB_NAME,REGION,STATUS
El Salvador,2026-07-18T23:00:41,Primary manufacturing facility located in El Salvador.,FAB001,El Salvador Fab,Central America,ACTIVE
United States,2026-07-18T23:00:41,High-volume semiconductor manufacturing facility.,FAB002,Alaska Fab,North America,ACTIVE
China,2026-07-18T23:00:41,Assembly and test manufacturing facility.,FAB003,China Fab,Asia Pacific,ACTIVE
Japan,2026-07-18T23:00:41,Advanced wafer fabrication facility.,FAB004,Japan Fab,Asia,ACTIVE


root
 |-- COUNTRY: string (nullable = true)
 |-- CREATED_DATE: string (nullable = true)
 |-- FAB_DESC: string (nullable = true)
 |-- FAB_ID: string (nullable = true)
 |-- FAB_NAME: string (nullable = true)
 |-- REGION: string (nullable = true)
 |-- STATUS: string (nullable = true)

Spark DataFrame record count: 4


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6648127967175669>, line 169
    157 # ============================================================
    158 # 11. WRITE TO UNITY CATALOG BRONZE TABLE
    159 # ============================================================
    161 TARGET_TABLE = "mes.bronze.fms_fab"
    164 (
    165     df_fms_fab
    166     .write
    167     .format("delta")
    168     .mode("overwrite")
--> 169     .saveAsTable(TARGET_TABLE)
    170 )
    173 print(
    174     f"✅ Data successfully written to: {TARGET_TABLE}"
    175 )

File <command-6648127967175669>, line 129
    120 print(
    121     "✅ Data successfully received from Supabase REST API"
    122 )
    125 # ============================================================
    126 # 7. CREATE SPARK DATAFRAME
    127 # ============================================================
--> 129 df